# Detección de anomalías administrativas con DBSCAN

## Caso a resolver

El área de administración procesa facturas y gastos de distintos tipos: compras rutinarias, servicios recurrentes y adquisiciones especiales. Se requiere encontrar registros que no pertenezcan a ningún patrón normal para priorizar revisión de documentación, autorización y presupuesto.

Usaremos DBSCAN de scikit-learn. DBSCAN identifica grupos densos y marca como ruido los registros aislados. No utiliza Isolation Forest, Local Outlier Factor ni SVM.

## Objetivos

- Construir facturas sintéticas con variables administrativas realistas.
- Explicar densidad, vecindarios, puntos núcleo, frontera y ruido.
- Identificar facturas aisladas.
- Visualizar clusters y anomalías.
- Evaluar el detector con etiquetas sintéticas.
- Interpretar las alertas para un proceso administrativo.

## 1. Contexto administrativo

### Situación

Administración suele recibir facturas de naturaleza diferente. Una factura de papelería no debe compararse directamente con una compra de equipo o con un servicio mensual. Por eso es útil reconocer varios patrones normales en lugar de imponer una única media global.

### Pregunta de negocio

¿Qué facturas o gastos deben revisarse porque no parecen pertenecer a ningún grupo normal de operación?

Las posibles causas incluyen error de captura, factura duplicada, proveedor inusual, pago fuera de plazo, desviación presupuestal, gasto sin autorización o una operación legítima pero excepcional.

### Técnica elegida: DBSCAN

DBSCAN agrupa puntos que tienen suficientes vecinos dentro de una distancia. Los puntos que no pertenecen a ninguna región suficientemente densa se etiquetan como ruido. En este notebook, el ruido será una señal de anomalía administrativa.

## 1.1 Principios de DBSCAN

DBSCAN utiliza dos parámetros:

- eps: radio máximo del vecindario.
- min_samples: número mínimo de observaciones dentro del vecindario para considerar que existe densidad.

Tipos de puntos:

- Punto núcleo: tiene al menos min_samples vecinos dentro de eps.
- Punto frontera: está cerca de un punto núcleo, aunque no tenga suficientes vecinos propios.
- Ruido: no alcanza densidad suficiente ni está conectado a un núcleo.

El algoritmo expande clusters conectando puntos núcleo cercanos. No es necesario indicar cuántos clusters existen previamente.

En scikit-learn, la etiqueta -1 representa ruido. En este ejercicio convertiremos esa etiqueta en es_anomalia=1.

### Consecuencia importante

DBSCAN detecta rareza por densidad. Un grupo pequeño pero compacto puede ser considerado un cluster normal. Una factura aislada dentro del espacio administrativo se marca como ruido. Por eso eps y min_samples deben calibrarse con datos históricos y conocimiento del proceso.

## 2. Configuración

Importamos herramientas de generación, tablas, escalamiento, DBSCAN, métricas y visualización. La semilla garantiza reproducibilidad en Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

### Explicación detallada

numpy genera valores numéricos y controla la semilla. pandas construye la tabla de facturas. StandardScaler iguala las escalas antes de calcular distancias. DBSCAN identifica regiones densas. Las métricas se usan sólo para evaluar el ejercicio sintético.

## 3. Generación de datos administrativos realistas

Creamos tres patrones normales:

1. Gastos rutinarios: importes pequeños, pocos conceptos y pago rápido.
2. Servicios recurrentes: importes medios, más conceptos y pago mensual.
3. Compras especiales: importes altos, más conceptos y plazos mayores.

Después agregamos facturas anómalas con combinaciones dispersas: importes muy altos o muy bajos, plazos inusuales, impuestos atípicos, muchos conceptos y desviación presupuestal alta.

In [ ]:
n_por_cluster = 400
columnas = ['importe_factura_mxn', 'dias_hasta_pago', 'numero_conceptos',
            'impuesto_pct', 'frecuencia_proveedor_dias', 'desviacion_presupuesto_pct']

def crear_cluster(n, media, desviaciones):
    return rng.normal(loc=media, scale=desviaciones, size=(n, len(columnas)))

rutina = crear_cluster(n_por_cluster, [850, 12, 3, 16, 30, 4], [180, 4, 1, 1.0, 8, 4])
recurrente = crear_cluster(n_por_cluster, [5200, 30, 7, 16, 30, 6], [700, 5, 1.5, 1.0, 7, 5])
especial = crear_cluster(n_por_cluster, [22000, 48, 13, 16, 60, 8], [2500, 7, 2, 1.0, 12, 6])
normales = np.vstack([rutina, recurrente, especial])

anomalias = np.column_stack([
    rng.uniform(80, 60000, 60),
    rng.uniform(0, 150, 60),
    rng.integers(1, 45, 60),
    rng.choice([0, 5, 16, 25, 30], 60),
    rng.uniform(1, 365, 60),
    rng.uniform(-40, 80, 60)
])

df_normales = pd.DataFrame(normales, columns=columnas)
df_anomalias = pd.DataFrame(anomalias, columns=columnas)
df_normales['es_anomalia_real'] = 0
df_anomalias['es_anomalia_real'] = 1
df = pd.concat([df_normales, df_anomalias], ignore_index=True)
df['importe_factura_mxn'] = df['importe_factura_mxn'].clip(50, None)
df['dias_hasta_pago'] = df['dias_hasta_pago'].clip(0, None)
df['numero_conceptos'] = df['numero_conceptos'].round().clip(1, None).astype(int)
df['impuesto_pct'] = df['impuesto_pct'].clip(0, 100)
df['frecuencia_proveedor_dias'] = df['frecuencia_proveedor_dias'].clip(1, None)
df.insert(0, 'factura_id', [f'F-{i:05d}' for i in range(1, len(df)+1)])
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
variables = columnas
print(f'Facturas: {len(df):,} | Anomalías sintéticas: {df.es_anomalia_real.sum():,}')
display(df.head())

### Explicación detallada de la generación

Cada cluster representa una rutina administrativa legítima. El importe, los días de pago y el número de conceptos cambian según el tipo de gasto. El impuesto suele concentrarse en 16%, mientras la frecuencia del proveedor distingue compras recurrentes de proveedores ocasionales.

Las anomalías se generan con valores positivos y rangos administrativos plausibles. La desviación presupuestal puede ser negativa cuando el gasto queda por debajo del presupuesto y positiva cuando lo excede. No se debe eliminar automáticamente: es una señal para revisión.

## 4. Validación de datos

Revisamos faltantes, estadísticos y reglas básicas: importes no negativos, días de pago válidos, conceptos enteros, impuestos entre 0% y 100% y frecuencia positiva.

In [ ]:
print('Valores faltantes:')
print(df[variables].isna().sum())
display(df[variables].describe().round(2))
print('Controles administrativos:')
print({
    'importes_invalidos': int((df.importe_factura_mxn <= 0).sum()),
    'dias_pago_invalidos': int((df.dias_hasta_pago < 0).sum()),
    'conceptos_invalidos': int((df.numero_conceptos < 1).sum()),
    'impuestos_fuera_rango': int(((df.impuesto_pct < 0) | (df.impuesto_pct > 100)).sum()),
    'frecuencias_invalidas': int((df.frecuencia_proveedor_dias <= 0).sum())
})

### Interpretación de la validación

Los controles deben devolver cero. El objetivo es que una alerta de DBSCAN represente una rareza del comportamiento administrativo y no una unidad mal formada. La desviación presupuestal se conserva aunque sea negativa porque describe relación con el presupuesto, no un error físico.

## 4.1 Distribuciones

Los histogramas muestran cómo se distribuyen las variables y permiten observar los tres patrones normales y las colas de las facturas atípicas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, variable in zip(axes.ravel(), variables):
    sns.histplot(data=df, x=variable, hue='es_anomalia_real', bins=35, kde=True,
                 palette={0:'#4C78A8', 1:'#E45756'}, alpha=.45, ax=ax)
    ax.set_title(variable)
plt.tight_layout()
plt.show()

### Interpretación

El importe y la frecuencia pueden tener colas derechas. Los histogramas ayudan a saber si una sola escala domina visualmente y si las anomalías están concentradas en extremos. En producción conviene realizar este análisis por proveedor, departamento y periodo.

## 4.2 Relaciones administrativas

Observamos relaciones entre importe y desviación presupuestal, importe y días de pago, y frecuencia del proveedor e importe.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (x, y) in zip(axes, [('importe_factura_mxn','desviacion_presupuesto_pct'),
                             ('importe_factura_mxn','dias_hasta_pago'),
                             ('frecuencia_proveedor_dias','importe_factura_mxn')]):
    sns.scatterplot(data=df, x=x, y=y, hue='es_anomalia_real',
                    palette={0:'#4C78A8', 1:'#E45756'}, alpha=.65, ax=ax)
    ax.set_title(f'{x} vs {y}')
plt.tight_layout()
plt.show()

### Interpretación de relaciones

Una factura grande puede tener una desviación presupuestal moderada si corresponde a una compra planificada. En cambio, una factura grande con desviación positiva extrema y proveedor poco frecuente puede ser un caso prioritario. DBSCAN utiliza la combinación completa de variables, no una regla individual.

## 4.3 Matriz de correlación y resumen por etiqueta

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlación entre variables administrativas')
plt.show()
display(df.groupby('es_anomalia_real')[variables].agg(['median', 'mean', 'std', 'max']).round(2))

### Interpretación

La matriz resume relaciones lineales, mientras DBSCAN trabaja con distancias en el espacio completo. El resumen permite contrastar la escala de operaciones normales y anomalías sintéticas, pero las diferencias no deben convertirse directamente en reglas de rechazo.

## 5. Preparación y escalamiento

DBSCAN depende de distancias. Estandarizamos las variables para que importe, días, conceptos, porcentaje y frecuencia tengan una influencia comparable.

In [ ]:
X = df[variables].copy()
y_real = df['es_anomalia_real'].to_numpy()
escalador = StandardScaler()
X_escalada = escalador.fit_transform(X)
print('Matriz:', X_escalada.shape)
print('Medias escaladas:', X_escalada.mean(axis=0).round(3))
print('Desviaciones escaladas:', X_escalada.std(axis=0).round(3))

### Explicación detallada

X contiene sólo variables explicativas. factura_id y la etiqueta no entran al modelo. StandardScaler transforma cada columna a una escala comparable. En producción, el escalador debe ajustarse con un periodo de referencia y reutilizarse para periodos nuevos.

## 6. Entrenamiento con DBSCAN

Probamos un radio eps y un mínimo de vecinos. No se indica el número de clusters: DBSCAN los descubre a partir de densidad.

In [ ]:
modelo = DBSCAN(eps=0.85, min_samples=12)
etiquetas = modelo.fit_predict(X_escalada)
df['cluster'] = etiquetas
df['prediccion_anomalia'] = (df['cluster'] == -1).astype(int)
print('Clusters encontrados:', sorted(df.loc[df.cluster >= 0, 'cluster'].unique()))
print(f'Registros de ruido: {df.prediccion_anomalia.sum():,} ({df.prediccion_anomalia.mean():.1%})')
display(df.groupby('cluster').size().rename('facturas').sort_index())

### Explicación detallada del entrenamiento

DBSCAN asigna una etiqueta a cada factura. Las etiquetas no representan una jerarquía ni un orden: son identificadores de grupos. El valor -1 significa ruido. Los grupos positivos pueden representar rutinas como compras pequeñas, servicios recurrentes o adquisiciones de mayor importe.

La cantidad de ruido debe interpretarse contra la capacidad de revisión. Un eps muy pequeño produce demasiados casos aislados; uno muy grande puede unir patrones distintos y ocultar rarezas.

## 7. Evaluación

Comparamos el ruido detectado con la etiqueta sintética. Aunque en producción normalmente no existiría una etiqueta completa, aquí permite validar el ejercicio.

In [ ]:
y_pred = df['prediccion_anomalia']
precision = precision_score(y_real, y_pred)
recall = recall_score(y_real, y_pred)
print(f'Precisión: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
print(classification_report(y_real, y_pred, target_names=['normal', 'anomalía'], digits=3))
cm = confusion_matrix(y_real, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. normal', 'Pred. anomalía'],
            yticklabels=['Real normal', 'Real anomalía'])
plt.title('Matriz de confusión de DBSCAN')
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.show()

### Interpretación detallada

Un verdadero positivo es una factura aislada que efectivamente pertenece al grupo anómalo sintético. Un falso positivo es un gasto legítimo que quedó fuera de una región densa; puede ser una compra especial válida. Un falso negativo es una anomalía que quedó cerca de un grupo normal.

La precisión estima la eficiencia de la cola de revisión y el recall estima la cobertura de anomalías. En administración, ambos deben complementarse con evidencia documental y reglas de autorización.

## 8. Visualización de clusters y ruido

Separamos dos preguntas para evitar una leyenda ambigua:

1. **¿A qué grupo pertenece la factura?** En el panel izquierdo, el color representa el cluster. `Grupo normal 0` y `Grupo normal 1` son grupos descubiertos; `Ruido DBSCAN` significa que el registro no pertenece a una región densa.
2. **¿Debe revisarse?** En el panel derecho, el color representa únicamente el estado de alerta. Gris significa sin alerta y rojo significa alerta DBSCAN.

Así, los colores no mezclan cluster y predicción. Es posible que una factura pertenezca a un cluster normal y aun así sea revisada por otra regla; por eso conviene analizar ambas vistas por separado.

In [ ]:
df['nombre_cluster'] = df['cluster'].map({
    -1: 'Ruido DBSCAN',
     0: 'Grupo normal 0',
     1: 'Grupo normal 1'
})
df['estado_alerta'] = df['prediccion_anomalia'].map({
    0: 'Sin alerta',
    1: 'Alerta DBSCAN'
})

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Panel 1: los colores representan únicamente los clusters descubiertos.
colores_cluster = {
    'Grupo normal 0': '#4C78A8',
    'Grupo normal 1': '#2CA02C',
    'Ruido DBSCAN': '#D62728'
}
for nombre, grupo in df.groupby('nombre_cluster'):
    axes[0].scatter(grupo['importe_factura_mxn'], grupo['dias_hasta_pago'],
                    label=nombre, color=colores_cluster[nombre], alpha=.70, s=38)
axes[0].set_xscale('log')
axes[0].set_title('Clusters descubiertos por DBSCAN')
axes[0].set_xlabel('Importe de factura (MXN, escala logarítmica)')
axes[0].set_ylabel('Días hasta el pago')
axes[0].legend(title='Cluster')

# Panel 2: los colores representan únicamente el estado de alerta.
colores_alerta = {'Sin alerta': '#7F7F7F', 'Alerta DBSCAN': '#D62728'}
for estado, grupo in df.groupby('estado_alerta'):
    axes[1].scatter(grupo['importe_factura_mxn'], grupo['dias_hasta_pago'],
                    label=estado, color=colores_alerta[estado], alpha=.75, s=42)
axes[1].set_xscale('log')
axes[1].set_title('Facturas marcadas para revisión')
axes[1].set_xlabel('Importe de factura (MXN, escala logarítmica)')
axes[1].set_ylabel('Días hasta el pago')
axes[1].legend(title='Estado')

plt.tight_layout()
plt.show()

### Interpretación visual

Los clusters densos representan rutinas administrativas. Los marcadores de ruido son candidatos para revisión. La escala logarítmica del importe permite ver simultáneamente facturas pequeñas y grandes; no cambia los datos usados previamente, sólo facilita la visualización.

## 9. Interpretación de las alertas

Agregamos señales descriptivas para orientar la revisión documental. Estas reglas explican la alerta, pero no reemplazan DBSCAN.

In [ ]:
alertas = df[df.prediccion_anomalia == 1].copy()
alertas['senal_administrativa'] = np.select(
    [alertas.desviacion_presupuesto_pct > 30,
     alertas.dias_hasta_pago > 90,
     alertas.impuesto_pct.isin([0, 5, 25, 30]),
     alertas.numero_conceptos > 25,
     alertas.frecuencia_proveedor_dias > 180],
    ['desviación presupuestal alta', 'pago fuera de plazo habitual',
     'impuesto atípico', 'muchos conceptos', 'proveedor poco frecuente'],
    default='combinación administrativa inusual')
display(alertas.sort_values('importe_factura_mxn', ascending=False)
        [['factura_id'] + variables + ['cluster', 'senal_administrativa']]
        .head(15).round(2))
display(alertas.senal_administrativa.value_counts().rename_axis('señal').to_frame('alertas'))

### Interpretación operativa

Cada alerta debe comprobarse contra orden de compra, recepción, autorización, contrato, proveedor, centro de costo y presupuesto. Una desviación alta puede ser válida si existe autorización; un impuesto atípico puede corresponder a una operación exenta. La detección prioriza la revisión, no determina incumplimiento.

## 10. Sensibilidad de eps y min_samples

DBSCAN es sensible a ambos parámetros. Probamos combinaciones y medimos cuántos registros se clasifican como ruido, además de precisión y recall.

In [ ]:
resultados = []
for eps in [.60, .85, 1.10]:
    for minimo in [8, 12, 20]:
        m = DBSCAN(eps=eps, min_samples=minimo)
        etiquetas_temp = m.fit_predict(X_escalada)
        p = (etiquetas_temp == -1).astype(int)
        resultados.append({'eps': eps, 'min_samples': minimo, 'ruido': p.sum(),
                           'precision': precision_score(y_real, p),
                           'recall': recall_score(y_real, p)})
sensibilidad = pd.DataFrame(resultados)
display(sensibilidad.style.format({'precision':'{:.1%}', 'recall':'{:.1%}'}))

### Interpretación de sensibilidad

Un eps pequeño exige mayor cercanía y puede convertir muchos registros legítimos en ruido. Un eps grande conecta regiones y puede ocultar casos aislados. Un min_samples alto exige grupos más densos y reduce clusters pequeños.

La configuración debe elegirse con datos históricos, volumen de revisión y costo de errores. No se debe escoger sólo la fila con mayor recall si genera más alertas de las que el equipo puede revisar.

## 11. Conclusiones

- DBSCAN permite detectar facturas que no pertenecen a ningún patrón denso.
- Es adecuado cuando administración tiene varios tipos normales de gasto y no quiere fijar el número de grupos previamente.
- El escalamiento es indispensable porque las variables usan unidades distintas.
- El ruido debe interpretarse como candidato a revisión, no como fraude o incumplimiento automático.
- eps y min_samples controlan directamente el equilibrio entre cobertura y carga administrativa.
- Las etiquetas sintéticas sirven para aprendizaje; una implementación real requiere auditoría humana y validación temporal.

### Plan recomendado

1. Consolidar históricos de facturas, órdenes de compra, pagos y presupuestos.
2. Separar por tipo de gasto, proveedor y departamento cuando existan rutinas muy distintas.
3. Calibrar DBSCAN con revisiones documentales.
4. Integrar las alertas con autorización y trazabilidad.
5. Medir falsos positivos, falsos negativos y ahorro de tiempo de auditoría.

## 12. Resumen reproducible

In [ ]:
print({'facturas': len(df), 'anomalias_reales': int(y_real.sum()),
       'registros_ruido': int(y_pred.sum()),
       'precision': round(precision, 3),
       'recall': round(recall, 3)})